# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/quangptt0910/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

- One row = one pseudonymized warehouse (client, content) observation with id (client_hash_id and content_hash_id)
- Timw window: observed over a 30-day prior window (2026-03) used to predict what happens to that same item in the following 30-day window (2026-04)
- One thing deliberately excluded: Any data from April 2026 is strictly excluded from feature (it is the label window)

In [1]:
%pip -q install duckdb
import duckdb
import os, getpass

# Connect to DuckDB
con = duckdb.connect()
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")


Note: you may need to restart the kernel to use updated packages.


In [2]:
FEAT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {FEAT}
    GROUP BY 1,2,3 HAVING c > 1 LIMIT 5
""").df()

print("Grain duplicates:")
print(grain_check)

window_check = con.sql(f"""
    SELECT MIN(report_date) AS earliest_date,
           MAX(report_date) AS latest_date,
           COUNT(*) AS total_rows
    FROM {FEAT}
""").df()

print("Date window check:")
print(window_check)

availability_check = con.sql(f"""
    SELECT COUNT(*) AS rows_total,
           SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS rows_gsc,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS rows_ga4
    FROM {FEAT}
""").df()

print("Availability check:")
print(availability_check)

Grain duplicates:
Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, c]
Index: []
Date window check:
  earliest_date latest_date  total_rows
0    2026-03-01  2026-03-31     9841378
Availability check:
   rows_total   rows_gsc  rows_ga4
0     9841378  3611061.0  413966.0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

#### Features (safe to use before prediction):
- imp_march = SUM(gsc_impressions) over month=2026-03 — knowable on 2026-03-31.
- clk_march = SUM(gsc_clicks) over month=2026-03 — same reasoning.
- active_days_march = COUNT(*) FILTER (WHERE gsc_impressions > 0) over March.
- avg_pos_march = AVG(gsc_avg_position) over March.
- pos_vol_march = STDDEV_POP(gsc_avg_position) over March.
- content_type (joined from dim_content on content_hash_id).
- search_volume (joined from dim_content on content_hash_id).
- has_gsc_full_march = 1 if gsc_data_available IS TRUE for the client's full March history — knowable on 2026-03-31.
#### Label / proxy:
- is_declining_proxy = (imp_april < 0.8 * imp_march) AND (imp_march >= 100), computed only from April + March totals. Defined on a future window relative to the features.
#### Context (grouping / joins / splits only, never features):
- client_hash_id, content_hash_id, report_date, url_hash_id, keyword_hash_id.
#### Excluded:
- imp_april, clk_april, any April aggregate — label period, would leak.
- trend_direction, trend_pct — computed from the same window, not a future outcome.
- provider_used, model_used — product/metadata, not user signals (per docs/data-dictionary.md:46-47).
- visible_queries, rare_share, anon_share from fact_content_query_90d — the table's fixed 90-day window spans March + April, overlapping the label period.
- impressions_90d, clicks_90d, any 90-day rollup from the query table — same overlap.
- *_last30 aggregates whose tail crosses into April — same overlap.
- health_score, priority_score, action_type — product decision flags, not in the data and would be circular if rebuilt.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
FEAT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"
LBL  = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet')"
DIM_CONTENT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"

# Q1: GRAIN — expect 0
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {FEAT} GROUP BY 1,2,3 HAVING c > 1 LIMIT 5
""").df()
print("Grain duplicates (expect 0):")
print(grain_check)

# Q2: COUNTS + DATE SPAN — March
print("March window:")
print(con.sql(f"""
    SELECT MIN(report_date) first_d, MAX(report_date) last_d, COUNT(*) n_rows,
           COUNT(DISTINCT client_hash_id) n_clients,
           COUNT(DISTINCT content_hash_id) n_content
    FROM {FEAT}
""").df())

# Q2b: COUNTS + DATE SPAN — April
print("April window:")
print(con.sql(f"""
    SELECT MIN(report_date) first_d, MAX(report_date) last_d, COUNT(*) n_rows,
           COUNT(DISTINCT client_hash_id) n_clients,
           COUNT(DISTINCT content_hash_id) n_content
    FROM {LBL}
""").df())

# Q3: AVAILABILITY — IS TRUE filter
print("Availability March:")
print(con.sql(f"""
    SELECT COUNT(*) rows_total,
           SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) rows_gsc,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) rows_ga4
    FROM {FEAT}
""").df())

# Q4: PER-CATEGORY missingness on content_type (from dim_content)
print("Per-content_type missingness in dim_content:")
print(con.sql(f"""
    SELECT content_type,
           COUNT(*) AS n_items,
           AVG(CASE WHEN search_volume IS NULL THEN 1.0 ELSE 0 END) AS pct_missing_sv
    FROM {DIM_CONTENT}
    GROUP BY content_type
    ORDER BY n_items DESC
""").df())

# Q5: PANEL UNBALANCE — full distribution
print("Panel history depth (clients with full March):")
print(con.sql(f"""
    SELECT
      SUM(CASE WHEN start_d <= DATE '2026-03-01' AND end_d >= DATE '2026-03-31' THEN 1 ELSE 0 END) AS full_march_clients,
      SUM(CASE WHEN start_d  > DATE '2026-03-01' OR  end_d < DATE '2026-03-31' THEN 1 ELSE 0 END) AS partial_march_clients
    FROM (
      SELECT client_hash_id, MIN(report_date) start_d, MAX(report_date) end_d
      FROM {FEAT} GROUP BY 1
    )
""").df())

# Q6: LABEL COMPUTABILITY — pairs in March that also have April history
print("Label computability (March→April pairs):")
print(con.sql(f"""
    SELECT
      COUNT(DISTINCT f.client_hash_id || '|' || f.content_hash_id) AS march_items,
      COUNT(DISTINCT l.client_hash_id || '|' || l.content_hash_id) AS april_items,
      COUNT(DISTINCT CASE WHEN l.client_hash_id IS NOT NULL
                          THEN f.client_hash_id || '|' || f.content_hash_id END) AS march_items_with_april
    FROM (SELECT DISTINCT client_hash_id, content_hash_id FROM {FEAT}) f
    LEFT JOIN (SELECT DISTINCT client_hash_id, content_hash_id FROM {LBL}) l
      USING (client_hash_id, content_hash_id)
""").df())

# Q7: MINIMUM-HISTORY FILTER (policy threshold) — active_days >= 10
print("After active_days_march >= 10 filter:")
print(con.sql(f"""
    WITH m AS (
      SELECT client_hash_id, content_hash_id,
             COUNT(*) FILTER (WHERE gsc_impressions > 0) AS active_days_march
      FROM {FEAT}
      WHERE gsc_data_available IS TRUE
      GROUP BY 1,2
    )
    SELECT
      COUNT(*) AS items_total,
      SUM(CASE WHEN active_days_march >= 10 THEN 1 ELSE 0 END) AS items_kept
    FROM m
""").df())

Grain duplicates (expect 0):
Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, c]
Index: []
March window:
     first_d     last_d   n_rows  n_clients  n_content
0 2026-03-01 2026-03-31  9841378         55     331437
April window:
     first_d     last_d    n_rows  n_clients  n_content
0 2026-04-01 2026-04-30  10424730         61     362172
Availability March:
   rows_total   rows_gsc  rows_ga4
0     9841378  3611061.0  413966.0
Per-content_type missingness in dim_content:
         content_type  n_items  pct_missing_sv
0     keyword article   459174        0.186411
1      feedly article    57024        1.000000
2  comparison article     3408        0.000880
Panel history depth (clients with full March):
   full_march_clients  partial_march_clients
0                51.0                    4.0
Label computability (March→April pairs):
   march_items  april_items  march_items_with_april
0       331437       331436                  331436
After active_days_march >= 10 f

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

1. **Unbalanced panel (history depth).** Per-client history differs wildly. Section 3 Q5 shows how many clients have full March vs partial. We filter to clients with `gsc_data_start <= 2026-03-01`.
2. **GSC-only early history.** Rows before a client's `ga4_data_start` have GA4 columns zero-filled with `ga4_data_available = FALSE`. We filter on the flag, never treat zero as "no engagement."
3. **Query-table window overlap (leakage).** `fact_content_query_90d` covers a fixed 90-day window that spans March + April. We exclude `visible_queries`, `rare_share`, `anon_share`, and any 90-day rollup from features.
4. **Mid-month client onboards.** Clients with `gsc_data_start` after 2026-03-01 have partial March. Filtered out via (1).
5. **Content registered mid-window.** Items in `dim_content` with `content_created_at > 2026-03-01` cannot carry a clean April label. Filtered in feature build.
6. **April = label window only.** Features use `month=2026-03` exclusively; any aggregate whose tail crosses into April is excluded.
7. **Snapshot cutoff.** Daily facts stop at 2026-06-30; April is fully covered, so a 30-day April label is honest.
8. **Causal limit.** This data can rank candidates for review; it cannot prove a refresh caused a recovery. Decision-support only.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.